In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyID string"
        #     ",SourceSystem string"
        #     ",PortfolioKey string"
        #     ",PortfolioName string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",CDI string"
        #     ",RepresentedByPartyKey string"
        #     ",RepresentedByPartyName string"
        #     ",RepresentedByPartyType string"
        #     ",PartyKey string"
        #     ",PartyRepresentKey string"
        #     ",PartyTypeCode string"
        #     ",PartyType string"
        #     ",PartyStatusCode string"
        #     ",PartyStatus string"
        #     ",PartyStatusDate date"
        #     ",L3869StatusCode string"
        #     ",L3869Status string"
        #     ",L3869CourtDate timestamp"
        #     ",OCWStatus string"
        #     ",DeceasedFlag boolean"
        #     ",L3869InterbankingStatusCode int"
        #     ",L3869InterBankingStatus string"
        #     ",LPSCode int"
        #     ",LPS string"
        #     ",LPSDate timestamp"
        #     ",Watchlist string"
        #     ",FirstNameGR string"
        #     ",FirstNameENG string"
        #     ",LastNameGR string"
        #     ",LastNameENG string"
        #     ",FirstFatherNameGR string"
        #     ",FirstFatherNameENG string"
        #     ",LastFatherNameGR string"
        #     ",LastFatherNameENG string"
        #     ",FirstMotherNameGR string"
        #     ",LastMotherNameGR string"
        #     ",LastMotherNameENG string"
        #     ",LegalNameGR string"
        #     ",LegalNameENG string"
        #     ",PrimaryCommercialGR string"
        #     ",PrimaryCommercialENG string"
        #     ",SalutationTypeCode string"
        #     ",SalutationType string"
        #     ",GenderCode string"
        #     ",Gender string"
        #     ",MaritalStatusCode string"
        #     ",MaritalStatus string"
        #     ",CitizenshipCode string"
        #     ",Citizenship string"
        #     ",NationalityCode string"
        #     ",Nationality string"
        #     ",OccupationCode string"
        #     ",Occupation string"
        #     ",BirthDate string"
        #     ",BirthPlace string"
        #     ",DeathDate date"
        #     ",CommunicationLanguageCode int"
        #     ",CommunicationLanguage string"
        #     ",TaxRegistrationNumber string"
        #     ",TaxResidenceCountry string"
        #     ",OrganizationOriginCountry string"
        #     ",PassportNumber string"
        #     ",PassportIssuingAuthority string"
        #     ",PassportIssueDate date"
        #     ",PassportExpiredDate date"
        #     ",NationalNumber string"
        #     ",NationalNumberIssuingAuthority string"
        #     ",NationalNumberIssueDate date"
        #     ",NationalNumberExpiredDate date"
        #     ",MilitaryNumber string"
        #     ",MilitaryAuthority string"
        #     ",MilitaryIssueDate date"
        #     ",MilitaryExpiredDate date"
        #     ",SocialSecurityNumber string"
        #     ",ChildrenCertification string"
        #     ",ChildrenCertificationIssuingAuthority string"
        #     ",EnterpriseStartDate string"
        #     ",EnterpriseEndDate date"
        #     ",EconomicActivityCode int"
        #     ",EconomicActivity string"
        #     ",BranchActivityCode int"
        #     ",BranchActivity string"
        #     ",NumberOfChildren string"
        #     ",GeneralCommercialRegister string"
        #     ",NumberofEmployees string"
        #     ",TotalIncome string"
        #     ",BalanceSheetTotalAmount string"
        #     ",AnnualTurnoverAmount string"
        #     ",GroupCode string"
        #     ",GroupName string"
        #     ",GroupCodePortfolio int"
        #     ",PartyBankruptcyStage int"
        #     ",CovidFlag string"
        #     ",CovidCategoryCode int"
        #     ",CovidCategory string"
        #     ",AffectedCategoryCode int"
        #     ",AffectedCategory string"
        #     ",MonitoringUnitCode string"
        #     ",MonitoringUnit string"
        #     ",CreditRatingTypeCode int"
        #     ",CreditRatingType string"
        #     ",EducationLevelCode int"
        #     ",EducationLevel string"
        #     ",BalanceCategoryCode string"
        #     ",BalanceCategory string"
        #     ",ExtraSegmentCode string"
        #     ",ExtraSegment string"
        #     ",SegmentCode string"
        #     ",Segment string"
        #     ",SubSegmentCode string"
        #     ",SubSegment string"
        #     ",LegalFormCode int"
        #     ",LegalForm string"
        #     ",IsWholeshaleFlag boolean"
        #     ",EmploymentStatusCode int"
        #     ",EmploymentStatus string"
        #     ",EmploymentStatusDate date"
        #     ",CreatedBy string"
        #     ",CreatedByDate timestamp"
        #     ",LastUpdatedBy string"
        #     ",LastUpdatedByDate timestamp"
        #     ",ResubmissionStatus string"
        #     ",N4605Flag int"
        #     ",ConnectedBorrowerFlag boolean"
        #     ",StaffFlag string"
        #     ",L3869BorrowerHasOver50KBalance boolean"
        #     ",TargetLegalActionCode int"
        #     ",TargetLegalAction string"
        #     ",SubServicer string"
        #     ",RiskDescription string"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Party"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "prefix",
            "portfolioname",
            "vendor_code_qualco",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ========================== party_qualco_cepal ==========================
        party_qualco_cepal_extra_filters = None

        if party_qualco_cepal_extra_filters is not None:
            party_qualco_cepal_extra_filters.append(date_filter)
            party_qualco_cepal_final_filters = " AND ".join(
                party_qualco_cepal_extra_filters
            )

        else:
            party_qualco_cepal_final_filters = date_filter

        party_qualco_cepal_selected_fields = [
            "vendor_code",
            "cust_code",
            "cdi",
            "customer_type",
            "deceased_indicator",
            "customerstatus",
            "first_name",
            "last_name",
            "father_name",
            "mother_name",
            "gender",
            "marital_status_code",
            "nationality",
            "occupation",
            "birth_date",
            "tax_number",
            "doc_id_num",
            "doc_id_type",
            "bankruptcystage",
            "covid_affected_id",
            "balance_category",
            "extra_segment",
            "segment",
            "sub_segment",
            "legal_process_status_id",
            "legal_process_status_date",
            "affected_by",
            "targetlegalaction",
            "SpecialCases", # MM added
            "businessdate",
        ]

        party_qualco_cepal = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",
            party_qualco_cepal_final_filters,
            party_qualco_cepal_selected_fields,
        )

        # =================== lps_priority =======================
        lps_priority_extra_filters = None

        if lps_priority_extra_filters is not None:
            lps_priority_extra_filters.append(date_filter)
            lps_priority_final_filters = " AND ".join(lps_priority_extra_filters)
        else:
            lps_priority_final_filters = date_filter

        lps_priority_selected_fields = ["lps", "lps_code"]

        lps_priority = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qc_lps_priority",
            lps_priority_final_filters,
            lps_priority_selected_fields,
        )

        lps_priority = lps_priority.distinct()
        # ================= LST_OF_VAL ===========================
        lst_of_val_extra_filters = None

        if lst_of_val_extra_filters is not None:
            lst_of_val_extra_filters.append(date_filter)
            lst_of_val_final_filters = " AND ".join(lst_of_val_extra_filters)
        else:
            lst_of_val_final_filters = date_filter

        lst_of_val_selected_fields = ["lov_comments", "lovt_code", "lov_code", "lovt_desc", "lov_desc"]

        lst_of_val = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qc_lst_of_val",
            lst_of_val_final_filters,
            lst_of_val_selected_fields,
        )

        lst_of_val_1 = lst_of_val
        lst_of_val_2 = (
            lst_of_val_1.withColumnRenamed("lov_comments", "lov_comments_2")
            .withColumnRenamed("lov_code", "lov_code_2")
            .withColumnRenamed("lovt_desc", "lovt_desc_2")
        )

        lst_of_val_3 = (
            lst_of_val.withColumnRenamed("lov_comments", "target_legal_action_comments")
            .withColumnRenamed("lov_code", "target_legal_action_code")
            .withColumnRenamed("lovt_desc", "target_legal_action_tdesc")
            .withColumnRenamed("lov_desc", "target_legal_action_desc")
            .withColumnRenamed("lovt_code", "target_legal_action_lovt_code")
        )

        lst_of_val_4 = (
            lst_of_val.withColumnRenamed("lov_code", "lov_code_SubServicer")
            .withColumnRenamed("lovt_code", "lovt_code_SubServicer")
            .withColumnRenamed("lov_desc", "SubServicer")
            .withColumnRenamed("lov_comments", "lov_comments_SubServicer")
            .withColumnRenamed("lovt_desc", "lovt_desc_SubServicer")
        )

        # Joins
        party_qualco_cepal_df = (
            party_qualco_cepal.join(
                portfolio,
                party_qualco_cepal.vendor_code == portfolio.vendor_code_qualco,
                "left",
            )
            .join(
                lps_priority,
                lps_priority.lps_code == party_qualco_cepal.legal_process_status_id,
                "left",
            )
            .join(
                lst_of_val,
                (lst_of_val.lov_code == party_qualco_cepal.covid_affected_id)
                & (lst_of_val.lovt_desc.isin("CUSTOMER_DEDICATED_PROPERTY_01")),
                "left",
            )
            .join(
                lst_of_val_2,
                (lst_of_val_2.lov_code_2 == party_qualco_cepal.affected_by)
                & (lst_of_val_2.lovt_desc_2.isin("CUSTOMER_DEDICATED_PROPERTY_03")),
                "left",
            )
            .join(
                lst_of_val_3,
                (lst_of_val_3.target_legal_action_code == party_qualco_cepal.targetlegalaction)
                & (lst_of_val_3.target_legal_action_tdesc.isin("CUSTOMER_DEDICATED_PROPERTY_02")),
                "left",
            )
            .join(
                lst_of_val_4,
                ((lst_of_val_4.lov_code_SubServicer == party_qualco_cepal.SpecialCases)
                & (lst_of_val_4.lovt_code_SubServicer == 764)
                & (lst_of_val_4.SubServicer == "EUCLID DV01 - ASSIGN FOR SERVICING")),
            "left",
            )
        )

        # Hash Key
        party_qualco_cepal_primary_key_fields = [
            party_qualco_cepal.cust_code,
            lit("qc"),
        ]

        party_qualco_cepal_df = (
            party_qualco_cepal_df.withColumn(
                "PartyID",
                sha2(concat_ws(";", *party_qualco_cepal_primary_key_fields), 256),
            )
            .withColumn("RepresentativePartyId", lit(None).cast("string"))
            .withColumn("RepresentativePartyType", lit(None).cast("string"))
            .withColumn("PartyKeyPRF", lit(None).cast("int"))
            .withColumn("PartyType", lit(None).cast("string"))
            .withColumn(
                "PartyStatus",
                when(
                    party_qualco_cepal.deceased_indicator == 1, "deceased"
                ).otherwise("active"),
            )
            .withColumn("PartyStatusDate", lit(None).cast("date"))
            .withColumn(
                "PassportNumber",
                when(
                    party_qualco_cepal.doc_id_type == "ΔΙΑΒΑΤΗΡΙΟ",
                    party_qualco_cepal.doc_id_num,
                ).otherwise(""),
            )
            .withColumn(
                "PassportIssuingAuthority",
                when(
                    party_qualco_cepal.doc_id_type == "ΔΙΑΒΑΤΗΡΙΟ",
                    party_qualco_cepal.doc_id_type,
                ).otherwise(""),
            )
            .withColumn(
                "NationalNumber",
                when(
                    party_qualco_cepal.doc_id_type.isin(
                        "ΤΑΥΤΟΤΗΤΑ",
                        "ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΣ) - MONIMOY",
                        "ΤΑΥΤΟΤΗΤΑ ΕΥΡΩΠΑΪΚΗΣ ΕΝΩΣΗΣ",
                        "ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΝ) - MONIMOY",
                        "ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ ΕΛΛΗΝΙΚΗ ΑΣΤΥΝΟΜΙΑ",
                        "ΑΣΤ.ΤΑΥΤΟΤΗΤΑ",
                        "ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΑ) - MONIMOY",
                        "ΤΑΥΤ.ΣΤΡ./Σ.Α.",
                        "ΤΑΥΤΟΤΗΤΑ ΟΜΟΓΕΝΩΝ",
                        "ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ ΛΙΜΕΝΙΚΟ",
                        "ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ",
                        "ΠΥΡΟΣΒΕΣΤΙΚΗ",
                        "ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΣ) - ΕΦΕΔΡΟΥ",
                    ),
                    party_qualco_cepal.doc_id_num,
                ).otherwise(""),
            )
            .withColumn(
                "MaritalStatusCode",
                when(
                    party_qualco_cepal.marital_status_code.isNotNull(),
                    party_qualco_cepal.marital_status_code,
                )
                .otherwise("")
                .cast("string"),
            )
            .withColumn("FirstNameENG", lit(None).cast("string"))
            .withColumn("LastNameENG", lit(None).cast("string"))
            .withColumn("LegalNameGR", lit(None).cast("string"))
            .withColumn("PrimaryCommercialGR", lit(None).cast("string"))
            .withColumn("SalutationTypeCode", lit(None).cast("string"))
            .withColumn("SalutationType", lit(None).cast("string"))
            .withColumn("Citizenship", lit(None).cast("string"))
            .withColumn("CommunicationLanguage", lit(None).cast("string"))
            .withColumn("NationalIDIssueDate", lit(None).cast("date"))
            .withColumn("EnterpriseEndDate", lit(None).cast("date"))
            .withColumn("EconomicActivityCode", lit(None).cast("int"))
            .withColumn("EconomicActivity", lit(None).cast("string"))
            .withColumn("EducationLevel", lit(None).cast("string"))
            .withColumn("BranchActivityCode", lit(None).cast("int"))
            .withColumn("BranchActivity", lit(None).cast("string"))
            .withColumn("EducationLevel", lit(None).cast("string"))
            .withColumn("EducationLevelCode", lit(None).cast("int"))
            .withColumn("LegalRepresentativeId", lit(None).cast("int"))
            .withColumn("MonitoringUnitCode", lit(None).cast("int"))
            .withColumn("CreditRatingTypeCode", lit(None).cast("int"))
            .withColumn("EmploymentStatus", lit(None).cast("int"))
            .withColumn("PartyKeyART", lit(None).cast("string"))
            .withColumn("MilitaryID", lit(None).cast("string"))
            .withColumn("MilitaryAuthority", lit(None).cast("string"))
            .withColumn("ChildrenCertification", lit(None).cast("string"))
            .withColumn("LegalFormCode", lit(None).cast("int"))
            .withColumn("LegalForm", lit(None).cast("string"))
            .withColumn(
                "ChildrenCertificationIssuingAuthority", lit(None).cast("string")
            )
            .withColumn("TotalIncome", lit(None).cast("string"))
            .withColumn("LastMotherNameGR", lit(None).cast("string"))
            .withColumn("SocialSecurityNumber", lit(None).cast("string"))
            .withColumn("MaritalStatus", lit(None).cast("string"))
            .withColumn("PartyType", lit(None).cast("string"))
            .withColumn("EnterpriseStartDate", lit(None).cast("date"))
            .withColumn(
                "NationalNumberIssuingAuthority",
                when(
                    party_qualco_cepal.doc_id_type.isin(
                        "ΤΑΥΤΟΤΗΤΑ",
                        "ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΣ) - MONIMOY",
                        "ΤΑΥΤΟΤΗΤΑ ΕΥΡΩΠΑΪΚΗΣ ΕΝΩΣΗΣ",
                        "ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΝ) - MONIMOY",
                        "ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ ΕΛΛΗΝΙΚΗ ΑΣΤΥΝΟΜΙΑ",
                        "ΑΣΤ.ΤΑΥΤΟΤΗΤΑ",
                        "ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΑ) - MONIMOY",
                        "ΤΑΥΤ.ΣΤΡ./Σ.Α.",
                        "ΤΑΥΤΟΤΗΤΑ ΟΜΟΓΕΝΩΝ",
                        "ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ ΛΙΜΕΝΙΚΟ",
                        "ΤΑΥΤΟΤΗΤΑ ΣΩΜΑΤΩΝ ΑΣΦΑΛΕΙΑΣ",
                        "ΠΥΡΟΣΒΕΣΤΙΚΗ",
                        "ΤΑΥΤΟΤΗΤΑ ΕΝΟΠΛΩΝ ΔΥΝΑΜΕΩΝ (ΓΕΣ) - ΕΦΕΔΡΟΥ",
                    ),
                    party_qualco_cepal.doc_id_type,
                )
                .otherwise("")
                .cast("string"),
            )
            .withColumn(
                "CovidFlag",
                when(party_qualco_cepal.covid_affected_id.isNotNull(), "Y")
                .otherwise("N")
                .cast("int"),
            )
        )

        # ================== Selection of Final Columns ==================

        party_qualco_cepal_final = party_qualco_cepal_df.select(
            party_qualco_cepal_df.PartyID.alias("PartyID"),
            lit("qc").cast("string").alias("SourceSystem"),
            party_qualco_cepal.vendor_code.cast("string").alias("PortfolioKey"),
            portfolio.portfolioname.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            party_qualco_cepal.cdi.cast("string").alias("CDI"),
            lit(None).cast("string").alias("RepresentedByPartyKey"),
            lit(None).cast("string").alias("RepresentedByPartyName"),
            lit(None).cast("string").alias("RepresentedByPartyType"),
            party_qualco_cepal.cust_code.cast("string").alias("PartyKey"),
            party_qualco_cepal.cust_code.cast("string").alias("PartyRepresentKey"),
            party_qualco_cepal.customer_type.cast("string").alias("PartyTypeCode"),
            party_qualco_cepal.customer_type.cast("string").alias("PartyType"),
            lit(None)
            .cast("string")
            .alias("PartyStatusCode"),  # party_qualco_cepal.deceased_indicator
            lit(None)
            .cast("string")
            .alias("PartyStatus"),  # party_qualco_cepal_df.PartyStatus
            lit(None).cast("date").alias("PartyStatusDate"),
            lit(None).cast("string").alias("L3869StatusCode"),
            lit(None).cast("string").alias("L3869Status"),
            lit(None).cast("timestamp").alias("L3869CourtDate"),
            lit(None).alias("OCWStatus"),
            party_qualco_cepal.deceased_indicator.cast("boolean").alias(
                "DeceasedFlag"
            ),
            lit(None).cast("int").alias("L3869InterbankingStatusCode"),
            lit(None).cast("string").alias("L3869InterbankingStatus"),
            party_qualco_cepal_df.legal_process_status_id.cast("int").alias(
                "LPSCode"
            ),
            party_qualco_cepal_df.lps.cast("string").alias("LPS"),
            party_qualco_cepal_df.legal_process_status_date.alias("LPSDate"),
            lit(None).alias("Watchlist"),
            party_qualco_cepal.first_name.cast("string").alias("FirstNameGR"),
            lit(None).cast("string").alias("FirstNameENG"),
            party_qualco_cepal.last_name.cast("string").alias("LastNameGR"),
            lit(None).cast("string").alias("LastNameENG"),
            party_qualco_cepal.father_name.cast("string").alias(
                "FirstFatherNameGR"
            ),
            lit(None).cast("string").alias("FirstFatherNameENG"),
            lit(None).cast("string").alias("LastFatherNameGR"),
            lit(None).cast("string").alias("LastFatherNameENG"),
            lit(None).cast("string").alias("FirstMotherNameGR"),
            party_qualco_cepal.mother_name.cast("string").alias("LastMotherNameGR"),
            lit(None).cast("string").alias("LastMotherNameENG"),
            lit(None).cast("string").alias("LegalNameGR"),
            lit(None).cast("string").alias("LegalNameENG"),
            lit(None).cast("string").alias("PrimaryCommercialGR"),
            lit(None).cast("string").alias("PrimaryCommercialENG"),
            lit(None).cast("string").alias("SalutationTypeCode"),
            lit(None).cast("string").alias("SalutationType"),
            party_qualco_cepal.gender.cast("string").alias("GenderCode"),
            lit(None).cast("string").alias("Gender"),
            party_qualco_cepal_df.MaritalStatusCode.cast("string").alias(
                "MaritalStatusCode"
            ),
            lit(None).cast("string").alias("MaritalStatus"),
            lit(None).cast("string").alias("Citizenship"),
            lit(None).cast("string").alias("CitizenshipCode"),
            party_qualco_cepal.nationality.cast("string").alias("NationalityCode"),
            party_qualco_cepal.nationality.cast("string").alias("Nationality"),
            party_qualco_cepal.occupation.cast("string").alias("OccupationCode"),
            party_qualco_cepal.occupation.cast("string").alias("Occupation"),
            party_qualco_cepal.birth_date.cast("string").alias("BirthDate"),
            lit(None).cast("string").alias("BirthPlace"),
            lit(None).cast("date").alias("DeathDate"),
            lit(None).cast("string").alias("CommunicationLanguage"),
            lit(None).cast("int").alias("CommunicationLanguageCode"),
            party_qualco_cepal.tax_number.cast("string").alias(
                "TaxRegistrationNumber"
            ),
            lit(None).cast("string").alias("TaxResidenceCountry"),
            lit(None).cast("string").alias("OrganizationOriginCountry"),
            party_qualco_cepal_df.PassportNumber.cast("string").alias(
                "PassportNumber"
            ),
            party_qualco_cepal_df.PassportIssuingAuthority.cast("string").alias(
                "PassportIssuingAuthority"
            ),
            lit(None).cast("date").alias("PassportIssueDate"),
            lit(None).cast("date").alias("PassportExpiredDate"),
            party_qualco_cepal_df.NationalNumber.cast("string").alias(
                "NationalNumber"
            ),
            party_qualco_cepal_df.NationalNumberIssuingAuthority.cast(
                "string"
            ).alias("NationalNumberIssuingAuthority"),
            lit(None).cast("date").alias("NationalNumberIssueDate"),
            lit(None).cast("date").alias("NationalNumberExpiredDate"),
            lit(None).cast("string").alias("MilitaryNumber"),
            lit(None).cast("string").alias("MilitaryAuthority"),
            lit(None).cast("date").alias("MilitaryIssueDate"),
            lit(None).cast("date").alias("MilitaryExpiredDate"),
            lit(None).cast("string").alias("SocialSecurityNumber"),
            lit(None).cast("string").alias("ChildrenCertification"),
            lit(None).cast("string").alias("ChildrenCertificationIssuingAuthority"),
            lit(None).cast("string").alias("EnterpriseStartDate"),
            lit(None).cast("date").alias("EnterpriseEndDate"),
            lit(None).cast("int").alias("EconomicActivityCode"),
            lit(None).cast("string").alias("EconomicActivity"),
            lit(None).cast("int").alias("BranchActivityCode"),
            lit(None).cast("string").alias("BranchActivity"),
            lit(None).cast("string").alias("NumberOfChildren"),
            lit(None).cast("string").alias("GeneralCommercialRegister"),
            lit(None).cast("string").alias("NumberofEmployees"),
            lit(None).cast("string").alias("TotalIncome"),
            lit(None).cast("string").alias("BalanceSheetTotalAmount"),
            lit(None).cast("string").alias("AnnualTurnoverAmount"),
            lit(None).cast("string").alias("GroupCode"),
            lit(None).cast("string").alias("GroupName"),
            lit(None).alias("GroupCodePortfolio"),
            party_qualco_cepal.bankruptcystage.cast("int").alias(
                "PartyBankruptcyStage"
            ),
            party_qualco_cepal_df.CovidFlag.cast("string").alias("CovidFlag"),
            party_qualco_cepal.covid_affected_id.cast("int").alias(
                "CovidCategoryCode"
            ),
            party_qualco_cepal_df.lov_comments.cast("string").alias(
                "CovidCategory"
            ),
            party_qualco_cepal_df.affected_by.cast("int").alias(
                "AffectedCategoryCode"
            ),
            party_qualco_cepal_df.lov_comments_2.alias("AffectedCategory"),
            lit(None).cast("string").alias("MonitoringUnitCode"),
            lit(None).cast("string").alias("MonitoringUnit"),
            lit(None).cast("int").alias("CreditRatingTypeCode"),
            lit(None).cast("string").alias("CreditRatingType"),
            lit(None).cast("int").alias("EducationLevelCode"),
            lit(None).cast("string").alias("EducationLevel"),
            party_qualco_cepal_df.balance_category.cast("string").alias(
                "BalanceCategoryCode"
            ),
            party_qualco_cepal_df.balance_category.cast("string").alias(
                "BalanceCategory"
            ),
            party_qualco_cepal.extra_segment.cast("string").alias(
                "ExtraSegmentCode"
            ),
            party_qualco_cepal.extra_segment.cast("string").alias("ExtraSegment"),
            party_qualco_cepal.segment.cast("string").alias("SegmentCode"),
            party_qualco_cepal.segment.cast("string").alias("Segment"),
            party_qualco_cepal.sub_segment.cast("string").alias("SubSegmentCode"),
            party_qualco_cepal.sub_segment.cast("string").alias("SubSegment"),
            lit(None).cast("int").alias("LegalFormCode"),
            lit(None).cast("string").alias("LegalForm"),
            lit(None).alias("IsWholeshaleFlag"),
            lit(None).cast("string").alias("EmploymentStatus"),
            lit(None).alias("EmploymentStatusCode"),
            lit(None).cast("date").alias("EmploymentStatusDate"),
            lit(None).cast("string").alias("CreatedBy"),
            lit(None).cast("timestamp").alias("CreatedByDate"),
            lit(None).cast("string").alias("LastUpdatedBy"),
            lit(None).cast("timestamp").alias("LastUpdatedByDate"),
            lit(None).alias("ResubmissionStatus"),
            lit(None).alias("N4605Flag"),
            lit(None).alias("ConnectedBorrowerFlag"),
            lit(None).alias("StaffFlag"),
            lit(None).alias("L3869BorrowerHasOver50KBalance"),
            party_qualco_cepal_df.targetlegalaction.cast("int").alias(
                "TargetLegalActionCode"
            ),
            party_qualco_cepal_df.target_legal_action_desc.cast("string").alias(
                "TargetLegalAction"
            ),
            party_qualco_cepal_df.SubServicer.cast("string").alias("SubServicer"),
            lit(None).alias("RiskDescription"),
            party_qualco_cepal.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        party_qualco_cepal_final.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_party_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)